# KALIA replay shard — carve the frozen corpus slice that makes continual learning work

CL-1. Replay only protects the past if the replayed tokens are ones the daily run would
otherwise *not* see. This kernel therefore does two things: it carves `replay.bin` from
the v2b `train.bin`, and it writes `replay_manifest.json` recording the exact token
ranges. Any future 'new data' feed must skip those ranges, or the replay is a no-op.

Runs on **CPU only** (`enable_gpu: false`), so it consumes no GPU quota.

Inputs: the output of `kalia-prep-v2b` (attached as a kernel source).

In [ ]:
!mkdir -p /kaggle/working/data
!ls -lh /kaggle/input/notebooks/subhajitlucky/kalia-prep-v2b/data/

In [ ]:
import glob
import shutil
import subprocess
import sys

def find(name):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    assert hits, f"{name} not found - attach the kalia-prep-v2b output"
    return hits[0]

train_bin = find("train.bin")
shutil.copy("make_replay_shard.py", "/kaggle/working/")
print("train.bin:", train_bin)

In [ ]:
subprocess.run(
    [sys.executable, "make_replay_shard.py",
     "--train-bin", train_bin,
     "--out-dir", "/kaggle/working/data",
     "--fraction", "0.10",
     "--blocks", "32",
     "--seed", "1337"],
    check=True,
)

In [ ]:
!ls -lh /kaggle/working/data/
!cat /kaggle/working/data/replay_manifest.json

Done. `replay.bin` plus `replay_manifest.json` are the kernel output.

Attach this run's output to the training kernel together with a **new-data shard that
excludes the manifest's ranges**, and train with `replay_prob: 0.10` and
`--replay-bin <this output>/replay.bin`.

Measure before and after with `tools/forgetting_probe.py` — that is the only way to know
whether the replay did anything.